In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np
import pandas as pd

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Katalog roboczy (notatnik): {notebook_dir}")
print(f"Katalog główny projektu: {project_root}")
print(f"Załadowano ścieżkę modułów: {src_path}")

In [ ]:
from hep_tracking.dataset import TrackDataset
from hep_tracking.config import KNNModelConfig
from hep_tracking.models import BaseKNN, ScipyCKDTree, SklearnKNN
from hep_tracking.benchmark import ANNBenchmarkRunner
from hep_tracking.plots import plot_3d_hits, plot_distance_distributions, plot_scaling

print("Zależności modułu hep_tracking załadowane pomyślnie!")

In [ ]:
%matplotlib inline

data_dir = project_root / "data"
sanity_dataset_path = data_dir / "dataset_hard_10k.npz"

if sanity_dataset_path.exists():
    dataset_sanity = TrackDataset.from_npz(sanity_dataset_path)
    
    print(f"Wczytano zbiór kontrolny: {sanity_dataset_path.name}")
    print(f"Kształt macierzy cech: {dataset_sanity.X.shape}")
    
    plot_3d_hits(dataset_sanity)
    plot_distance_distributions(dataset_sanity)
else:
    print(f"Nie znaleziono pliku {sanity_dataset_path.name}. Upewnij się, że wygenerowałeś dane (make generate).")

In [ ]:
from hep_tracking.models import NumpyBruteForce, ScipyCKDTree, SklearnKNN

models_configs = [
    KNNModelConfig("Numpy Brute Force", NumpyBruteForce, {}),
    KNNModelConfig("Scipy cKDTree", ScipyCKDTree, {"workers": -1}),
    KNNModelConfig("Sklearn KDTree", SklearnKNN, {"algorithm": "kd_tree"}),
    KNNModelConfig("Sklearn BallTree", SklearnKNN, {"algorithm": "ball_tree"})
]

try:
    from hep_tracking.models import CuPyBruteForce
    models_configs.insert(1, KNNModelConfig("CuPy Brute Force", CuPyBruteForce, {}))
except ImportError:
    print("Moduł 'cupy' nie został znaleziony. Pomijam model 'CuPy Brute Force'.")

target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
dataset_modes = ["easy", "hard"]
k_neighbors = 5

runner = ANNBenchmarkRunner(k_neighbors=k_neighbors, warmup_runs=1, num_runs=3)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

all_results = []

for mode in dataset_modes:
    print(f"\n{'='*45}")
    print(f" TRYB DANYCH: {mode.upper()}")
    print(f"{'='*45}")

    for size_label, size_val in target_sizes.items():
        filename = data_dir / f"dataset_{mode}_{size_label}.npz"
        
        if not filename.exists():
            print(f"\n[BRAK PLIKU] {filename.name} - Pomijam.")
            continue

        dataset = TrackDataset.from_npz(filename)
        print(f"\n Załadowano: {size_label} ({len(dataset)} hitów)")
        dataset_name = f"{mode}_{size_label}"

        exact_knn = ScipyCKDTree(workers=-1)
        exact_knn.fit(dataset.X)
        _, true_idx = exact_knn.kneighbors(dataset.X, k=k_neighbors)

        active_configs = models_configs
        if size_val > 100000:
            active_configs = [
                c for c in models_configs 
                if c.name not in ["Numpy Brute Force", "CuPy Brute Force", "Sklearn BallTree"]
            ]
            print("  > Pomijam modele Brute Force oraz BallTree (Zbyt wolne dla 1M)")

        df_size = runner.run(
            models_configs=active_configs, 
            datasets={dataset_name: dataset}, 
            ground_truth={dataset_name: true_idx}
        )
        
        df_size['Mode'] = mode
        df_size['Size_Label'] = size_label
        df_size['Size'] = size_val
        all_results.append(df_size)

print("\nBenchmark zakończony sukcesem!")
results_df = pd.concat(all_results, ignore_index=True)
display(results_df.head())

In [ ]:
for mode in dataset_modes:
    df_mode = results_df[results_df['Mode'] == mode]
    
    plot_scaling(
        df=df_mode,
        x_col="Size",
        y_col="Query_Time_s",
        title=f"Skalowalność kNN - tryb: {mode.upper()}",
        log_x=True,
        log_y=True
    )